# Telco Customer Churn — Dataset Analysis

**Dataset:** [IBM Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)  
**Size:** 7,043 customers, 21 columns  
**License:** Open  
**Relevance to Tasknova:** Churn prediction baseline, Renewal Risk Intelligence (KPI), Revenue at Risk scoring  

This notebook loads the dataset, runs sanity checks, and explores churn patterns, feature distributions, and correlations.

## 1. Setup & Installation

In [ ]:
!pip install -q datasets pandas matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 30)

## 2. Load Dataset

The IBM Telco Churn dataset is available through multiple sources. We try Hugging Face first, then fall back to a direct URL.

In [ ]:
# Try loading from Hugging Face datasets hub
try:
    from datasets import load_dataset
    ds = load_dataset("scikit-learn/churn-prediction", split="train")
    df = ds.to_pandas()
    print(f"Loaded from Hugging Face: {len(df):,} rows")
except Exception as e:
    print(f"Hugging Face load failed: {e}")
    # Fallback: load from raw GitHub mirror of the IBM dataset
    url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
    try:
        df = pd.read_csv(url)
        print(f"Loaded from IBM GitHub: {len(df):,} rows")
    except Exception as e2:
        print(f"GitHub load also failed: {e2}")
        print("Please download manually from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn")
        print("Place the CSV as 'Telco-Customer-Churn.csv' in the notebooks directory.")
        df = pd.read_csv("Telco-Customer-Churn.csv")

print(f"\nShape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

In [ ]:
# Standardize column names to lowercase
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(f"Standardized columns: {list(df.columns)}")

## 3. Sanity Checks

In [ ]:
print("=" * 60)
print("SANITY CHECK 1: Data types")
print("=" * 60)
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

In [ ]:
print("=" * 60)
print("SANITY CHECK 2: Missing values")
print("=" * 60)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing": missing, "pct": missing_pct})
print(missing_report[missing_report.missing > 0] if missing.sum() > 0 else "No nulls found.")

# Also check for blank strings and whitespace-only values
print("\nBlank/whitespace-only values:")
for col in df.select_dtypes(include='object').columns:
    blanks = (df[col].astype(str).str.strip() == '').sum()
    if blanks > 0:
        print(f"  {col}: {blanks} blank values")

In [ ]:
print("=" * 60)
print("SANITY CHECK 3: Duplicate records")
print("=" * 60)

n_dupes = df.duplicated().sum()
print(f"Exact duplicates: {n_dupes}")

# Check for duplicate customer IDs
id_col = None
for candidate in ['customerid', 'customer_id', 'id']:
    if candidate in df.columns:
        id_col = candidate
        break

if id_col:
    n_id_dupes = df[id_col].duplicated().sum()
    print(f"Duplicate customer IDs ('{id_col}'): {n_id_dupes}")
    print(f"Unique customers: {df[id_col].nunique():,} / {len(df):,}")

In [ ]:
print("=" * 60)
print("SANITY CHECK 4: Numeric column ranges and outliers")
print("=" * 60)

# Fix: TotalCharges is often read as string due to blank values
for col in ['totalcharges', 'total_charges']:
    if col in df.columns and df[col].dtype == 'object':
        df[col] = pd.to_numeric(df[col], errors='coerce')
        coerced_nulls = df[col].isna().sum()
        print(f"Fixed '{col}': converted to numeric ({coerced_nulls} values became NaN)")

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns: {numeric_cols}")
print("\n" + df[numeric_cols].describe().round(2).to_string())

# Check for negative values where they shouldn't be
for col in numeric_cols:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f"\nWARNING: '{col}' has {neg_count} negative values")

In [ ]:
print("=" * 60)
print("SANITY CHECK 5: Target variable (Churn) distribution")
print("=" * 60)

churn_col = None
for candidate in ['churn', 'churned', 'churn_label']:
    if candidate in df.columns:
        churn_col = candidate
        break

if churn_col:
    print(f"Churn column: '{churn_col}'")
    print(f"Unique values: {df[churn_col].unique()}")
    print(f"\nDistribution:")
    vc = df[churn_col].value_counts()
    for val, count in vc.items():
        print(f"  {val}: {count:,} ({count/len(df)*100:.1f}%)")
    
    # Class imbalance ratio
    majority = vc.max()
    minority = vc.min()
    print(f"\nClass imbalance ratio: {majority/minority:.2f}:1")
    if majority/minority > 3:
        print("NOTE: Significant class imbalance — consider SMOTE, class weights, or stratified sampling.")
else:
    print("No churn column found. Available columns:", list(df.columns))

In [ ]:
print("=" * 60)
print("SANITY CHECK 6: Categorical field cardinality")
print("=" * 60)

cat_cols = df.select_dtypes(include='object').columns.tolist()
if id_col and id_col in cat_cols:
    cat_cols.remove(id_col)

for col in cat_cols:
    n_unique = df[col].nunique()
    print(f"\n'{col}' ({n_unique} unique):")
    print(f"  {df[col].value_counts().to_dict()}")

## 4. Exploratory Analysis

In [ ]:
# Churn rate overview
if churn_col:
    fig, ax = plt.subplots(figsize=(6, 4))
    colors = ['#55A868', '#C44E52']
    df[churn_col].value_counts().plot(kind='bar', ax=ax, color=colors, edgecolor='white')
    ax.set_title('Churn Distribution')
    ax.set_xlabel('Churned')
    ax.set_ylabel('Count')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    for i, (val, count) in enumerate(df[churn_col].value_counts().items()):
        ax.text(i, count + 50, f"{count:,}\n({count/len(df)*100:.1f}%)", ha='center', fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# Numeric feature distributions split by churn
if churn_col and len(numeric_cols) > 0:
    # Encode churn as binary for analysis
    if df[churn_col].dtype == 'object':
        df['_churn_binary'] = (df[churn_col].str.lower().isin(['yes', '1', 'true', 'churn'])).astype(int)
    else:
        df['_churn_binary'] = df[churn_col].astype(int)
    
    plot_cols = [c for c in numeric_cols if c != id_col and df[c].nunique() > 5]
    n_plots = min(len(plot_cols), 6)
    
    if n_plots > 0:
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        axes = axes.flatten()
        
        for i, col in enumerate(plot_cols[:6]):
            for label, color in [(0, '#55A868'), (1, '#C44E52')]:
                subset = df[df['_churn_binary'] == label][col].dropna()
                label_name = 'Churned' if label == 1 else 'Retained'
                axes[i].hist(subset, bins=30, alpha=0.6, color=color, label=label_name, density=True)
            axes[i].set_title(col)
            axes[i].legend(fontsize=8)
        
        for j in range(n_plots, 6):
            axes[j].set_visible(False)
        
        fig.suptitle('Feature Distributions by Churn Status', fontsize=14, y=1.02)
        plt.tight_layout()
        plt.show()

In [ ]:
# Churn rate by categorical features
if churn_col and '_churn_binary' in df.columns:
    high_signal_cols = [c for c in cat_cols if 2 <= df[c].nunique() <= 10]
    n_plots = min(len(high_signal_cols), 8)
    
    if n_plots > 0:
        cols_per_row = 4
        n_rows = (n_plots + cols_per_row - 1) // cols_per_row
        fig, axes = plt.subplots(n_rows, cols_per_row, figsize=(18, 5 * n_rows))
        axes = np.array(axes).flatten()
        
        for i, col in enumerate(high_signal_cols[:n_plots]):
            churn_rate = df.groupby(col)['_churn_binary'].mean().sort_values(ascending=False)
            churn_rate.plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='white')
            axes[i].set_title(f'Churn Rate by {col}', fontsize=10)
            axes[i].set_ylabel('Churn Rate')
            axes[i].set_ylim(0, 1)
            axes[i].axhline(df['_churn_binary'].mean(), color='red', linestyle='--', alpha=0.7)
            axes[i].tick_params(axis='x', rotation=45)
            for j, (val, rate) in enumerate(churn_rate.items()):
                axes[i].text(j, rate + 0.02, f"{rate:.0%}", ha='center', fontsize=8)
        
        for j in range(n_plots, len(axes)):
            axes[j].set_visible(False)
        
        fig.suptitle('Churn Rate by Categorical Features (red line = overall average)', fontsize=13, y=1.02)
        plt.tight_layout()
        plt.show()

In [ ]:
# Correlation heatmap for numeric features
if len(numeric_cols) >= 3 and '_churn_binary' in df.columns:
    corr_cols = numeric_cols + ['_churn_binary']
    corr_matrix = df[corr_cols].corr()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, ax=ax, square=True, linewidths=0.5)
    ax.set_title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()
    
    # Features most correlated with churn
    churn_corr = corr_matrix['_churn_binary'].drop('_churn_binary').abs().sort_values(ascending=False)
    print("Features most correlated with churn:")
    for feat, corr in churn_corr.head(10).items():
        direction = '+' if corr_matrix.loc[feat, '_churn_binary'] > 0 else '-'
        print(f"  {feat}: {direction}{corr:.3f}")

In [ ]:
# Tenure analysis — key indicator for renewal risk
tenure_col = None
for candidate in ['tenure', 'tenure_months', 'months']:
    if candidate in df.columns:
        tenure_col = candidate
        break

if tenure_col and churn_col:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Tenure distribution by churn
    for label, color, name in [(0, '#55A868', 'Retained'), (1, '#C44E52', 'Churned')]:
        subset = df[df['_churn_binary'] == label][tenure_col].dropna()
        axes[0].hist(subset, bins=30, alpha=0.6, color=color, label=name, density=True)
    axes[0].set_title('Tenure Distribution by Churn Status')
    axes[0].set_xlabel('Tenure (months)')
    axes[0].set_ylabel('Density')
    axes[0].legend()
    
    # Churn rate by tenure bucket
    df['_tenure_bucket'] = pd.cut(df[tenure_col], bins=[0, 6, 12, 24, 48, 72, 200],
                                   labels=['0-6m', '6-12m', '1-2y', '2-4y', '4-6y', '6y+'])
    bucket_churn = df.groupby('_tenure_bucket', observed=True)['_churn_binary'].agg(['mean', 'count'])
    bucket_churn['mean'].plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
    axes[1].set_title('Churn Rate by Tenure Bucket')
    axes[1].set_ylabel('Churn Rate')
    axes[1].set_xlabel('Tenure')
    axes[1].tick_params(axis='x', rotation=0)
    for i, (rate, count) in enumerate(zip(bucket_churn['mean'], bucket_churn['count'])):
        axes[1].text(i, rate + 0.02, f"{rate:.0%}\n(n={count:,})", ha='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
else:
    print("No tenure column found.")

In [ ]:
# Revenue at risk — monthly charges of churned customers
charges_col = None
for candidate in ['monthlycharges', 'monthly_charges']:
    if candidate in df.columns:
        charges_col = candidate
        break

if charges_col and '_churn_binary' in df.columns:
    churned = df[df['_churn_binary'] == 1]
    retained = df[df['_churn_binary'] == 0]
    
    print("Revenue at Risk Analysis:")
    print(f"  Churned customers: {len(churned):,}")
    print(f"  Monthly revenue at risk: ${churned[charges_col].sum():,.0f}")
    print(f"  Avg monthly charge (churned): ${churned[charges_col].mean():.2f}")
    print(f"  Avg monthly charge (retained): ${retained[charges_col].mean():.2f}")
    print(f"  Annualized revenue at risk: ${churned[charges_col].sum() * 12:,.0f}")
    
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(retained[charges_col], bins=30, alpha=0.6, color='#55A868', label='Retained', density=True)
    ax.hist(churned[charges_col], bins=30, alpha=0.6, color='#C44E52', label='Churned', density=True)
    ax.set_title('Monthly Charges: Churned vs Retained')
    ax.set_xlabel('Monthly Charges ($)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Summary

**Findings from this analysis:**

| Check | Status |
|---|---|
| Missing values | See check 2 (TotalCharges often has blanks) |
| Duplicates | See check 3 |
| Numeric ranges | See check 4 |
| Class balance | See check 5 (~26% churn, manageable imbalance) |
| Feature correlations | See heatmap |
| Tenure vs churn | See tenure analysis |

**Relevance to Tasknova:**  
- This is a baseline churn prediction dataset — not B2B SaaS-specific but structurally similar  
- Tenure, contract type, and monthly charges are the strongest churn signals here, mapping to Tasknova's Renewal Risk Intelligence KPI  
- Revenue at Risk calculation directly maps to Tasknova's Revenue at Risk Score  
- For production: Tasknova will need to build its own churn dataset combining CRM deal data + conversation signals (sentiment trends, engagement drop-off, stakeholder disengagement)  
- This dataset can validate the scoring framework before real data is available